In [1]:
import pandas as pd

In [2]:
crm = pd.read_excel("AI-Revenue-Leakage-System/data/crm_data.xlsx")
billing = pd.read_excel("AI-Revenue-Leakage-System/data/billing_data.xlsx")
onboarding = pd.read_excel("AI-Revenue-Leakage-System/data/onboarding_data.xlsx")

crm.head()

,Customer_ID,Deal_Value,Deal_Close_Date,Status,Assigned_Rep
0,1,93810,2025-01-08,Closed Won,Rahul
1,2,42098,2025-01-15,Closed Won,Priya
2,3,21395,2025-02-07,Closed Won,Priya
3,4,88907,2025-01-02,Closed Won,Priya
4,5,64987,2025-01-15,Closed Won,Priya


In [3]:
merged = crm.merge(billing, on="Customer_ID", how="left")
merged = merged.merge(onboarding, on="Customer_ID", how="left")

merged.head()

,Customer_ID,Deal_Value,Deal_Close_Date,Status,Assigned_Rep,Invoice_ID,Invoice_Amount,Invoice_Date,Onboarding_Start_Date,Onboarding_Completion_Date
0,1,93810,2025-01-08,Closed Won,Rahul,NaN,NaN,NaT,2025-01-09,2025-01-17
1,2,42098,2025-01-15,Closed Won,Priya,INV-2,38065.0,2025-01-30,2025-01-16,2025-02-02
2,3,21395,2025-02-07,Closed Won,Priya,INV-3,19500.0,2025-02-11,2025-02-08,2025-02-24
3,4,88907,2025-01-02,Closed Won,Priya,INV-4,88907.0,2025-01-14,2025-01-03,2025-01-20
4,5,64987,2025-01-15,Closed Won,Priya,INV-5,64987.0,2025-01-29,2025-01-16,2025-01-16


In [4]:
risk_flags = []
severity_scores = []

for index, row in merged.iterrows():
    issues = []
    severity = 0
    
    if pd.isna(row["Invoice_Amount"]):
        issues.append("Missing Invoice")
        severity += 3
        
    elif row["Deal_Value"] != row["Invoice_Amount"]:
        issues.append("Billing Mismatch")
        severity += 2
    
    delay = (row["Onboarding_Completion_Date"] - row["Deal_Close_Date"]).days
    if delay > 10:
        issues.append("Onboarding Delay Risk")
        severity += 1
    
    risk_flags.append(", ".join(issues) if issues else "No Risk")
    severity_scores.append(severity)

merged["Risk_Flags"] = risk_flags
merged["Severity_Score"] = severity_scores

merged.head()

,Customer_ID,Deal_Value,Deal_Close_Date,Status,Assigned_Rep,Invoice_ID,Invoice_Amount,Invoice_Date,Onboarding_Start_Date,Onboarding_Completion_Date,Risk_Flags,Severity_Score
0,1,93810,2025-01-08,Closed Won,Rahul,NaN,NaN,NaT,2025-01-09,2025-01-17,Missing Invoice,3
1,2,42098,2025-01-15,Closed Won,Priya,INV-2,38065.0,2025-01-30,2025-01-16,2025-02-02,"Billing Mismatch, Onboarding Delay Risk",3
2,3,21395,2025-02-07,Closed Won,Priya,INV-3,19500.0,2025-02-11,2025-02-08,2025-02-24,"Billing Mismatch, Onboarding Delay Risk",3
3,4,88907,2025-01-02,Closed Won,Priya,INV-4,88907.0,2025-01-14,2025-01-03,2025-01-20,Onboarding Delay Risk,1
4,5,64987,2025-01-15,Closed Won,Priya,INV-5,64987.0,2025-01-29,2025-01-16,2025-01-16,No Risk,0


In [6]:
risk_cases = merged[merged["Severity_Score"] > 0]
risk_cases.to_excel("AI-Revenue-Leakage-System/reports/revenue_leakage_report.xlsx", index=False)

print("Report Generated Successfully")

Report Generated Successfully


In [10]:
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.units import inch

doc = SimpleDocTemplate("AI-Revenue-Leakage-System/reports/executive_summary_report.pdf")
elements = []

styles = getSampleStyleSheet()
text = f"""
Revenue Leakage Executive Summary

Total Deals: {len(crm)}
Risk Cases Identified: {len(risk_cases)}

Primary Risk Areas:
- Missing Invoices
- Billing Mismatch
- Onboarding Delays
"""

elements.append(Paragraph(text, styles["Normal"]))
elements.append(Spacer(1, 0.5 * inch))
doc.build(elements)

In [12]:
merged["AI_Recommendation"] = "Investigate billing alignment and correct system workflow."